# Task 01 | Nested Transaction Analyzer

Concepts: Data structures, sorting, comprehensions

Problem: Given a list of transaction dictionaries with keys id, customer, category, amount, and status, produce a report containing each customer’s successful spending, top category, and rank by spending. Ignore failed transactions and reject duplicate transaction IDs.

Requirements: Return a list sorted by spending descending, then customer name ascending. Customers with no successful transactions must appear with total 0. Do not use pandas.

In [1]:

transactions = [
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"}
]

seen = []
customers = {}
categories = {}

# Calculate spending
for t in transactions:

    if t["id"] in seen:
        continue
    seen.append(t["id"])

    name = t["customer"]

    if name not in customers:
        customers[name] = 0
        categories[name] = {}

    if t["status"] == "success":
        customers[name] += t["amount"]

        cat = t["category"]
        if cat not in categories[name]:
            categories[name][cat] = 0

        categories[name][cat] += t["amount"]


# Create report
report = []

for name in customers:
    top = None
    highest = 0

    for cat in categories[name]:
        if categories[name][cat] > highest:
            highest = categories[name][cat]
            top = cat

    report.append({
        "customer": name,
        "total_spending": customers[name],
        "top_category": top
    })


# Sort by spending and customer name
for i in range(len(report)):
    for j in range(i + 1, len(report)):
        if (report[i]["total_spending"] < report[j]["total_spending"] or
            (report[i]["total_spending"] == report[j]["total_spending"] and
             report[i]["customer"] > report[j]["customer"])):
            report[i], report[j] = report[j], report[i]


# Assign ranks
rank = 0
previous = None

for i in range(len(report)):
    if report[i]["total_spending"] != previous:
        rank = i + 1

    report[i]["rank"] = rank
    previous = report[i]["total_spending"]

print(report)


[{'customer': 'Sara', 'total_spending': 700, 'top_category': 'Food', 'rank': 1}, {'customer': 'Ali', 'total_spending': 500, 'top_category': 'Food', 'rank': 2}]


# Task 02 | Longest Valid Window

Concepts: Sliding window, dictionaries

Problem: Write longest_unique_window(text) to find the longest contiguous substring with no repeated characters. Return the substring, start index, and length.

Requirements: Target O(n) time. On equal-length windows, select the earliest. Treat spaces and punctuation as characters.
Sample / expected behavior: Input: "abcabcbb" -> ("abc", 0, 3); "" -> ("", 0, 0).

In [2]:

def longest_unique_window(text):

    seen = {}
    start = 0
    best_start = 0
    max_length = 0

    for i, char in enumerate(text):

        if char in seen and seen[char] >= start:
            start = seen[char] + 1

        seen[char] = i

        length = i - start + 1

        if length > max_length:
            max_length = length
            best_start = start

    substring = text[best_start:best_start + max_length]

    return substring, best_start, max_length


# Test cases
print(longest_unique_window("abcabcbb"))
print(longest_unique_window(""))
print(longest_unique_window("abcdef"))
print(longest_unique_window("pwwkew"))
print(longest_unique_window("aabbcc"))


('abc', 0, 3)
('', 0, 0)
('abcdef', 0, 6)
('wke', 2, 3)
('ab', 1, 2)


# Task 03 | Recursive JSON Path Explorer
Concepts: Recursion, nested lists/dictionaries

Saylani IT Training | Advanced Python Practical Examination

Problem: Implement find_paths(data, target) that recursively searches nested dictionaries and
lists and returns all paths whose value equals target.

Requirements: Use dot notation for dictionary keys and [index] for list positions. Preserve
traversal order; do not treat bool True as numeric 1.

Sample / expected behavior: Input: {"a":[{"x":5},5],"b":{"y":5}}, target=5 -> ["a[0].x", "a[1]", "b.y"].

In [23]:
def find_paths(data, target):
    paths = []

    def search(value, current_path):
        # Do not allow True to match 1, or False to match 0.
        if isinstance(value, bool) or isinstance(target, bool):
            matches = type(value) is type(target) and value == target
        else:
            matches = value == target and not isinstance(value, (dict, list))

        if matches:
            paths.append(current_path)

        if isinstance(value, dict):
            for key, child in value.items():
                new_path = str(key) if current_path == "" else current_path + "." + str(key)
                search(child, new_path)

        elif isinstance(value, list):
            for index, child in enumerate(value):
                new_path = current_path + "[" + str(index) + "]"
                search(child, new_path)

    search(data, "")
    return paths


if __name__ == "__main__":
    data = {"a": [{"x": 5}, 5], "b": {"y": 5}}
    print(find_paths(data, 5))  # ['a[0].x', 'a[1]', 'b.y']
    print(find_paths({"a": True, "b": 1}, 1))  # ['b']


['a[0].x', 'a[1]', 'b.y']
['b']


# Task 04 | Dependency Execution Planner

Concepts: Graphs, sets, algorithms

Problem: Given tasks and prerequisite pairs (A,B) meaning A must finish before B, return a valid
execution order.

Requirements: Detect cycles and raise ValueError with a clear message. For multiple available
tasks, select alphabetically first. Reject unknown task names.

Sample / expected behavior: Tasks=["build","test","deploy"];

dependencies=[("build","test"),("test","deploy")] -> ["build","test","deploy"].

In [24]:
def execution_order(tasks, dependencies):
    task_set = set(tasks)

    if len(task_set) != len(tasks):
        raise ValueError("Task names must be unique.")

    # Each task starts with zero prerequisites.
    incoming = {task: 0 for task in tasks}
    next_tasks = {task: [] for task in tasks}

    for before, after in dependencies:
        if before not in task_set or after not in task_set:
            raise ValueError("Unknown task name in dependencies.")
        next_tasks[before].append(after)
        incoming[after] += 1

    available = sorted([task for task in tasks if incoming[task] == 0])
    result = []

    while available:
        current = available.pop(0)
        result.append(current)

        for following in sorted(next_tasks[current]):
            incoming[following] -= 1
            if incoming[following] == 0:
                available.append(following)
                available.sort()

    if len(result) != len(tasks):
        raise ValueError("Cycle detected: tasks cannot be completed.")

    return result


if __name__ == "__main__":
    tasks = ["build", "test", "deploy"]
    dependencies = [("build", "test"), ("test", "deploy")]
    print(execution_order(tasks, dependencies))


['build', 'test', 'deploy']


# Task 05 | Expression Validator and Evaluator

Concepts: Stacks, parsing, exception handling

Problem: Create a calculator for integer expressions containing +, -, *, /, parentheses, and
whitespace. Apply standard operator precedence and parentheses.

Requirements: Do not use eval(), exec(), or third-party parsers. Handle unary minus, malformed
expressions, and division by zero.

Sample / expected behavior: Input: "-(3 + 2) * 4 + 18 / 3" -> -14.0.

In [25]:
# Task 05: Expression Validator and Evaluator
# Supports integers, +, -, *, /, parentheses, spaces, and unary +/-
# eval() and exec() are not used.

class ExpressionError(ValueError):
    pass


class Calculator:
    def __init__(self, text):
        self.text = text
        self.position = 0

    def skip_spaces(self):
        while self.position < len(self.text) and self.text[self.position].isspace():
            self.position += 1

    def current(self):
        self.skip_spaces()
        if self.position < len(self.text):
            return self.text[self.position]
        return None

    def eat(self, expected):
        if self.current() == expected:
            self.position += 1
            return True
        return False

    def parse_number_or_parentheses(self):
        if self.eat("("):
            value = self.parse_expression()
            if not self.eat(")"):
                raise ExpressionError("Missing closing parenthesis.")
            return value

        start = self.position
        while self.position < len(self.text) and self.text[self.position].isdigit():
            self.position += 1

        if start == self.position:
            raise ExpressionError("A number or '(' was expected.")

        return int(self.text[start:self.position])

    def parse_factor(self):
        if self.eat("+"):
            return self.parse_factor()
        if self.eat("-"):
            return -self.parse_factor()
        return self.parse_number_or_parentheses()

    def parse_term(self):
        value = self.parse_factor()
        while True:
            if self.eat("*"):
                value *= self.parse_factor()
            elif self.eat("/"):
                divisor = self.parse_factor()
                if divisor == 0:
                    raise ZeroDivisionError("Cannot divide by zero.")
                value /= divisor
            else:
                break
        return value

    def parse_expression(self):
        value = self.parse_term()
        while True:
            if self.eat("+"):
                value += self.parse_term()
            elif self.eat("-"):
                value -= self.parse_term()
            else:
                break
        return value

    def calculate(self):
        if not self.text.strip():
            raise ExpressionError("Expression cannot be empty.")

        result = self.parse_expression()
        self.skip_spaces()

        if self.position != len(self.text):
            raise ExpressionError("Invalid character or malformed expression.")

        return result


def evaluate(expression):
    return Calculator(expression).calculate()


if __name__ == "__main__":
    print(evaluate("-(3 + 2) * 4 + 18 / 3"))  # -14.0
    # Try: print(evaluate("10 / (2 + 3)"))


-14.0


# Task 06 | Banking Ledger with OOP

Concepts: Classes, encapsulation, properties
Problem: Build Account and Bank classes. Support deposit, withdrawal, transfer, and account
statements. Maintain immutable transaction records containing transaction ID, action, amount,
and resulting balance.

Requirements: Reject nonpositive amounts, insufficient funds, and nonexistent accounts. A
transfer must be atomic: both balances update or neither does.

Sample / expected behavior: Account A=1000, B=200; transfer 350 from A to B -> A=650,
B=550.

In [26]:
# Task 06: Banking Ledger with OOP

from dataclasses import dataclass
from uuid import uuid4


@dataclass(frozen=True)
class Transaction:
    transaction_id: str
    action: str
    amount: float
    resulting_balance: float


class Account:
    def __init__(self, account_id, balance=0):
        if balance < 0:
            raise ValueError("Opening balance cannot be negative.")
        self.account_id = account_id
        self._balance = balance
        self._transactions = []

    @property
    def balance(self):
        return self._balance

    @property
    def statement(self):
        return tuple(self._transactions)

    def _record(self, action, amount):
        record = Transaction(str(uuid4()), action, amount, self._balance)
        self._transactions.append(record)

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive.")
        self._balance += amount
        self._record("deposit", amount)

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive.")
        if amount > self._balance:
            raise ValueError("Insufficient funds.")
        self._balance -= amount
        self._record("withdrawal", amount)


class Bank:
    def __init__(self):
        self.accounts = {}

    def add_account(self, account):
        if account.account_id in self.accounts:
            raise ValueError("Account already exists.")
        self.accounts[account.account_id] = account

    def transfer(self, from_id, to_id, amount):
        if from_id not in self.accounts or to_id not in self.accounts:
            raise ValueError("Account does not exist.")
        if from_id == to_id:
            raise ValueError("Cannot transfer to the same account.")
        if amount <= 0:
            raise ValueError("Amount must be positive.")

        sender = self.accounts[from_id]
        receiver = self.accounts[to_id]

        if sender.balance < amount:
            raise ValueError("Insufficient funds.")

        # Validate everything first, then update both accounts.
        sender._balance -= amount
        receiver._balance += amount
        sender._record("transfer_out", amount)
        receiver._record("transfer_in", amount)


if __name__ == "__main__":
    bank = Bank()
    bank.add_account(Account("A", 1000))
    bank.add_account(Account("B", 200))
    bank.transfer("A", "B", 350)
    print(bank.accounts["A"].balance)  # 650
    print(bank.accounts["B"].balance)  # 550


650
550


# Task 07 | Employee Payroll Engine
Concepts: Inheritance, polymorphism, abstract classes

Problem: Define abstract Employee with calculate_pay(). Implement SalariedEmployee,
HourlyEmployee, and CommissionEmployee with different pay formulas. Generate a unified
payroll summary.

Requirements: Hourly overtime above 40 hours pays 1.5x. Commission employees receive base
+ sales*rate. Prevent direct instantiation of Employee.
Sample / expected behavior: Hourly rate=20, hours=45 -> 950; Commission base=500,
sales=4000, rate=.10 -> 900.

In [27]:
# Task 07: Employee Payroll Engine

from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        pass


class SalariedEmployee(Employee):
    def __init__(self, name, monthly_salary):
        super().__init__(name)
        self.monthly_salary = monthly_salary

    def calculate_pay(self):
        return self.monthly_salary


class HourlyEmployee(Employee):
    def __init__(self, name, hourly_rate, hours):
        super().__init__(name)
        self.hourly_rate = hourly_rate
        self.hours = hours

    def calculate_pay(self):
        regular_hours = min(self.hours, 40)
        overtime_hours = max(self.hours - 40, 0)
        return (regular_hours * self.hourly_rate +
                overtime_hours * self.hourly_rate * 1.5)


class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name)
        self.base = base
        self.sales = sales
        self.rate = rate

    def calculate_pay(self):
        return self.base + self.sales * self.rate


def payroll_summary(employees):
    summary = []
    for employee in employees:
        summary.append({
            "name": employee.name,
            "pay": employee.calculate_pay()
        })
    return summary


if __name__ == "__main__":
    staff = [
        HourlyEmployee("Ali", 20, 45),
        CommissionEmployee("Sara", 500, 4000, 0.10),
        SalariedEmployee("Ahmed", 3000)
    ]
    print(payroll_summary(staff))
    # Ali = 950; Sara = 900


[{'name': 'Ali', 'pay': 950.0}, {'name': 'Sara', 'pay': 900.0}, {'name': 'Ahmed', 'pay': 3000}]


# Task 08 | Cache Decorator with Expiration

Concepts: Decorators, closures, functools, time

Problem: Implement @ttl_cache(seconds) that caches results for repeated positional and
keyword arguments and recomputes after expiration. Track cache hits and misses.
Requirements: Preserve function metadata with functools.wraps. Handle equivalent keyword
argument order; document behavior for unhashable inputs. Use time.monotonic().

Sample / expected behavior: A call with x=2,y=3 followed by y=3,x=2 before expiry should
return a cache hit.

In [28]:
# Task 08: Cache Decorator with Expiration

from functools import wraps
from time import monotonic


def ttl_cache(seconds):
    if seconds < 0:
        raise ValueError("Cache time cannot be negative.")

    def decorator(function):
        cache = {}
        stats = {"hits": 0, "misses": 0}

        @wraps(function)
        def wrapper(*args, **kwargs):
            # Sorting keyword arguments makes their order irrelevant.
            key = (args, tuple(sorted(kwargs.items())))

            # Arguments must be hashable to be used as dictionary keys.
            try:
                hash(key)
            except TypeError:
                raise TypeError("Cached function arguments must be hashable.")

            now = monotonic()

            if key in cache:
                saved_time, saved_result = cache[key]
                if now - saved_time < seconds:
                    stats["hits"] += 1
                    return saved_result

            stats["misses"] += 1
            result = function(*args, **kwargs)
            cache[key] = (now, result)
            return result

        wrapper.cache_hits = lambda: stats["hits"]
        wrapper.cache_misses = lambda: stats["misses"]
        wrapper.clear_cache = cache.clear
        return wrapper

    return decorator


if __name__ == "__main__":
    @ttl_cache(10)
    def add(x, y):
        print("Calculating...")
        return x + y

    print(add(x=2, y=3))
    print(add(y=3, x=2))  # Cache hit
    print("Hits:", add.cache_hits())
    print("Misses:", add.cache_misses())


Calculating...
5
5
Hits: 1
Misses: 1


# Task 09 | Streaming Log Inspector

Concepts: Generators, iterators, file handling

Problem: Write a generator read_errors(path) that streams a log file line by line, yielding
(line_number, timestamp, message) only for ERROR records. Build summarize_errors() to count
errors by date.

Requirements: Do not load the whole file into memory. Skip malformed lines while tracking their
count. Expected format: YYYY-MM-DD HH:MM:SS | LEVEL | message.

Sample / expected behavior: Input line: 2026-11-07 10:30:00 | ERROR | Connection lost ->
(1,"2026-11-07 10:30:00","Connection lost").

In [29]:
# Task 09: Streaming Log Inspector

from collections import defaultdict


def read_errors(path):
    # The count is available after the generator has been fully consumed.
    read_errors.malformed_count = 0

    with open(path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            parts = line.strip().split("|", 2)

            if len(parts) != 3:
                read_errors.malformed_count += 1
                continue

            timestamp = parts[0].strip()
            level = parts[1].strip()
            message = parts[2].strip()

            # Check the timestamp format.
            try:
                from datetime import datetime
                datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
            except ValueError:
                read_errors.malformed_count += 1
                continue

            if not level or not message:
                read_errors.malformed_count += 1
                continue

            if level == "ERROR":
                yield line_number, timestamp, message


read_errors.malformed_count = 0


def summarize_errors(path):
    counts = defaultdict(int)

    for line_number, timestamp, message in read_errors(path):
        date = timestamp[:10]
        counts[date] += 1

    return dict(counts), read_errors.malformed_count


if __name__ == "__main__":
    # Create a small sample log file.
    sample_lines = [
        "2026-11-07 10:30:00 | ERROR | Connection lost\n",
        "2026-11-07 10:35:00 | INFO | Service started\n",
        "bad log line\n",
        "2026-11-08 09:00:00 | ERROR | Database unavailable\n"
    ]

    with open("sample.log", "w", encoding="utf-8") as file:
        file.writelines(sample_lines)

    for error in read_errors("sample.log"):
        print(error)

    print(summarize_errors("sample.log"))


(1, '2026-11-07 10:30:00', 'Connection lost')
(4, '2026-11-08 09:00:00', 'Database unavailable')
({'2026-11-07': 1, '2026-11-08': 1}, 1)


# Task 10 | CSV Inventory Reconciliation

Concepts: CSV, file handling, validation

Problem: Read inventory.csv (sku,name,quantity) and movements.csv (sku,type,units). Apply IN
and OUT movements in order, then write updated_inventory.csv and rejected_movements.csv.

Requirements: Reject unknown SKUs, negative or zero units, invalid movement types, and any
OUT causing negative stock. Continue processing after rejected rows.

Sample / expected behavior: SKU P01 starts at 10; OUT 4, OUT 9, IN 2 -> final 8; second
movement rejected.

In [30]:
# Task 10: CSV Inventory Reconciliation

import csv


def reconcile_inventory(inventory_file, movements_file,
                        updated_file="updated_inventory.csv",
                        rejected_file="rejected_movements.csv"):
    inventory = {}

    with open(inventory_file, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            sku = row["sku"].strip()
            inventory[sku] = {
                "sku": sku,
                "name": row["name"],
                "quantity": int(row["quantity"])
            }

    rejected = []

    with open(movements_file, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            sku = row.get("sku", "").strip()
            movement_type = row.get("type", "").strip().upper()

            try:
                units = int(row.get("units", ""))
                if units <= 0:
                    raise ValueError("Units must be greater than zero.")
                if sku not in inventory:
                    raise ValueError("Unknown SKU.")
                if movement_type not in ("IN", "OUT"):
                    raise ValueError("Movement type must be IN or OUT.")

                current_quantity = inventory[sku]["quantity"]

                if movement_type == "OUT" and units > current_quantity:
                    raise ValueError("Not enough stock for this OUT movement.")

                if movement_type == "IN":
                    inventory[sku]["quantity"] += units
                else:
                    inventory[sku]["quantity"] -= units

            except ValueError as error:
                row["reason"] = str(error)
                rejected.append(row)

    with open(updated_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["sku", "name", "quantity"])
        writer.writeheader()
        writer.writerows(inventory.values())

    rejected_fields = ["sku", "type", "units", "reason"]
    with open(rejected_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=rejected_fields)
        writer.writeheader()
        writer.writerows(rejected)

    return inventory, rejected


if __name__ == "__main__":
    # Prepare sample input files.
    with open("inventory.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["sku", "name", "quantity"])
        writer.writerow(["P01", "Pen", 10])

    with open("movements.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["sku", "type", "units"])
        writer.writerow(["P01", "OUT", 4])
        writer.writerow(["P01", "OUT", 9])
        writer.writerow(["P01", "IN", 2])

    print(reconcile_inventory("inventory.csv", "movements.csv"))


({'P01': {'sku': 'P01', 'name': 'Pen', 'quantity': 8}}, [{'sku': 'P01', 'type': 'OUT', 'units': '9', 'reason': 'Not enough stock for this OUT movement.'}])


# Task 11 | Multi-File Student Gradebook

Concepts: File handling, dictionaries, exceptions

Problem: Merge students.csv (student_id,name) and scores.csv (student_id,subject,marks).
Calculate per-student average, grade, and rank; export report.csv.
Requirements: Marks must be 0–100; reject unknown IDs and duplicate student-subject entries.
Grades: A>=85, B>=70, C>=55, D>=40, otherwise F. Rank by average, then ID.
Sample / expected behavior: Student 101 with marks 90,80,85 -> average 85.0, grade A.

In [31]:
# Task 11: Multi-File Student Gradebook

import csv


def get_grade(average):
    if average >= 85:
        return "A"
    elif average >= 70:
        return "B"
    elif average >= 55:
        return "C"
    elif average >= 40:
        return "D"
    return "F"


def create_gradebook(students_file, scores_file, output_file="report.csv"):
    students = {}

    with open(students_file, "r", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            students[row["student_id"]] = row["name"]

    marks_by_student = {}
    seen_subjects = set()

    with open(scores_file, "r", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            student_id = row["student_id"]
            subject = row["subject"]

            if student_id not in students:
                continue

            key = (student_id, subject)
            if key in seen_subjects:
                continue
            seen_subjects.add(key)

            try:
                marks = float(row["marks"])
            except ValueError:
                continue

            if not 0 <= marks <= 100:
                continue

            marks_by_student.setdefault(student_id, []).append(marks)

    report = []
    for student_id, name in students.items():
        marks = marks_by_student.get(student_id, [])
        if not marks:
            continue

        average = sum(marks) / len(marks)
        report.append({
            "student_id": student_id,
            "name": name,
            "average": round(average, 2),
            "grade": get_grade(average)
        })

    report.sort(key=lambda row: (-row["average"], row["student_id"]))

    for index, row in enumerate(report, start=1):
        row["rank"] = index

    with open(output_file, "w", newline="", encoding="utf-8") as file:
        fields = ["student_id", "name", "average", "grade", "rank"]
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(report)

    return report


if __name__ == "__main__":
    with open("students.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["student_id", "name"])
        writer.writerow(["101", "Ayesha"])

    with open("scores.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["student_id", "subject", "marks"])
        writer.writerow(["101", "English", 90])
        writer.writerow(["101", "Math", 80])
        writer.writerow(["101", "Science", 85])

    print(create_gradebook("students.csv", "scores.csv"))


[{'student_id': '101', 'name': 'Ayesha', 'average': 85.0, 'grade': 'A', 'rank': 1}]


## Task 12 | Custom Iterable Range Toolkit

Concepts: Iterators, generators, dunder methods

Problem: Create a ReversibleRange(start,stop,step) class supporting iteration, reversed
iteration, len(), and membership tests without constructing a list.

Requirements: Match Python range stop-exclusive semantics for positive and negative nonzero
steps. Implement __iter__, __reversed__, __len__, and __contains__.

Sample / expected behavior: ReversibleRange(2,11,3) -> [2,5,8]; reversed -> [8,5,2]; 5 in object
-> True.

In [32]:
# Task 12: Custom Iterable Range Toolkit

class ReversibleRange:
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("Step cannot be zero.")
        self.start = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        current = self.start

        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step
        else:
            while current > self.stop:
                yield current
                current += self.step

    def __reversed__(self):
        # range() calculates the same stop-exclusive values without a list.
        return reversed(range(self.start, self.stop, self.step))

    def __len__(self):
        return len(range(self.start, self.stop, self.step))

    def __contains__(self, value):
        return value in range(self.start, self.stop, self.step)


if __name__ == "__main__":
    numbers = ReversibleRange(2, 11, 3)
    print(list(numbers))          # [2, 5, 8]
    print(list(reversed(numbers)))  # [8, 5, 2]
    print(len(numbers))           # 3
    print(5 in numbers)            # True


[2, 5, 8]
[8, 5, 2]
3
True


# Task 13 | Booking Conflict Scheduler

Concepts: Datetime, sorting, interval algorithms

Problem: Implement a room booking scheduler. Each booking has room, start datetime, end
datetime, and booking ID. Accept bookings only when they do not overlap existing bookings for
the same room.

Requirements: Use datetime objects; reject end<=start and duplicate IDs. Adjacent bookings are
allowed. Return conflicts for rejected requests.

Sample / expected behavior: Room A: 09:00–10:00 and 10:00–11:00 accepted; 09:30–10:30
rejected.

In [33]:
# Task 13: Booking Conflict Scheduler

from datetime import datetime


class BookingScheduler:
    def __init__(self):
        self.bookings = []
        self.booking_ids = set()

    def add_booking(self, room, start, end, booking_id):
        if end <= start:
            return False, "End time must be after start time."

        if booking_id in self.booking_ids:
            return False, "Duplicate booking ID."

        for booking in self.bookings:
            same_room = booking["room"] == room
            overlaps = start < booking["end"] and end > booking["start"]

            if same_room and overlaps:
                return False, "Booking conflicts with an existing booking."

        self.bookings.append({
            "room": room,
            "start": start,
            "end": end,
            "booking_id": booking_id
        })
        self.booking_ids.add(booking_id)
        self.bookings.sort(key=lambda item: (item["room"], item["start"]))
        return True, "Booking accepted."


if __name__ == "__main__":
    scheduler = BookingScheduler()
    date = "2026-11-07 "

    start1 = datetime.strptime(date + "09:00", "%Y-%m-%d %H:%M")
    end1 = datetime.strptime(date + "10:00", "%Y-%m-%d %H:%M")
    start2 = datetime.strptime(date + "10:00", "%Y-%m-%d %H:%M")
    end2 = datetime.strptime(date + "11:00", "%Y-%m-%d %H:%M")
    start3 = datetime.strptime(date + "09:30", "%Y-%m-%d %H:%M")
    end3 = datetime.strptime(date + "10:30", "%Y-%m-%d %H:%M")

    print(scheduler.add_booking("A", start1, end1, "B1"))
    print(scheduler.add_booking("A", start2, end2, "B2"))
    print(scheduler.add_booking("A", start3, end3, "B3"))


(True, 'Booking accepted.')
(True, 'Booking accepted.')
(False, 'Booking conflicts with an existing booking.')


# Task 14 | Command-Line Contact Manager

Concepts: Modules, JSON, CRUD, validation

Problem: Build a menu-driven CLI contact manager with add, search, update, delete, list, and
persistent JSON storage. Each contact has a unique ID, name, phone, and email.

Requirements: Use separate functions/modules for storage and business logic. Handle
missing/corrupt JSON safely without silently deleting existing data. Search names caseinsensitively.

Sample / expected behavior: Add ID=1, name="Ayesha"; restart application; search "aye" ->
contact remains available.

In [ ]:

import json
import os

FILE_NAME = "contacts.json"


# Step 1: Load contacts from JSON file
def load_contacts():
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, "r") as file:
            return json.load(file)
    return []


# Step 2: Save contacts into JSON file
def save_contacts(contacts):
    with open(FILE_NAME, "w") as file:
        json.dump(contacts, file, indent=4)


# Step 3: Add a new contact
def add_contact(contacts):
    contact_id = input("Enter ID: ")

    for contact in contacts:
        if contact["id"] == contact_id:
            print("Contact ID already exists!")
            return

    name = input("Enter Name: ")
    phone = input("Enter Phone: ")
    email = input("Enter Email: ")

    if not name or not phone or not email:
        print("All fields are required!")
        return

    new_contact = {
        "id": contact_id,
        "name": name,
        "phone": phone,
        "email": email
    }

    contacts.append(new_contact)
    save_contacts(contacts)

    print("Contact added successfully!")


# Step 4: Search for a contact
def search_contact(contacts):
    name = input("Enter name to search: ").lower()

    found = False

    for contact in contacts:
        if name in contact["name"].lower():
            print(contact)
            found = True

    if not found:
        print("Contact not found!")


# Step 5: Update a contact
def update_contact(contacts):
    contact_id = input("Enter ID to update: ")

    for contact in contacts:
        if contact["id"] == contact_id:

            contact["name"] = input("Enter new name: ")
            contact["phone"] = input("Enter new phone: ")
            contact["email"] = input("Enter new email: ")

            save_contacts(contacts)
            print("Contact updated successfully!")
            return

    print("Contact not found!")


# Step 6: Delete a contact
def delete_contact(contacts):
    contact_id = input("Enter ID to delete: ")

    for contact in contacts:
        if contact["id"] == contact_id:

            contacts.remove(contact)
            save_contacts(contacts)

            print("Contact deleted successfully!")
            return

    print("Contact not found!")


# Step 7: Display all contacts
def display_contacts(contacts):
    if len(contacts) == 0:
        print("No contacts available!")
    else:
        for contact in contacts:
            print(contact)


# Step 8: Main menu
def main():

    try:
        contacts = load_contacts()
    except (json.JSONDecodeError, OSError):
        print("Error reading contacts file. Please check contacts.json.")
        return

    while True:

        print("\n===== CONTACT MANAGER =====")
        print("1. Add Contact")
        print("2. Search Contact")
        print("3. Update Contact")
        print("4. Delete Contact")
        print("5. Display All Contacts")
        print("6. Exit")

        choice = input("Enter your choice: ")

        if choice == "1":
            add_contact(contacts)

        elif choice == "2":
            search_contact(contacts)

        elif choice == "3":
            update_contact(contacts)

        elif choice == "4":
            delete_contact(contacts)

        elif choice == "5":
            display_contacts(contacts)

        elif choice == "6":
            print("Thank you for using Contact Manager!")
            break

        else:
            print("Invalid choice! Try again.")


# Run the program
if __name__ == "__main__":
    main()



===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. Update Contact
4. Delete Contact
5. Display All Contacts
6. Exit
Invalid choice! Try again.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. Update Contact
4. Delete Contact
5. Display All Contacts
6. Exit
Invalid choice! Try again.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. Update Contact
4. Delete Contact
5. Display All Contacts
6. Exit
Invalid choice! Try again.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. Update Contact
4. Delete Contact
5. Display All Contacts
6. Exit
Invalid choice! Try again.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. Update Contact
4. Delete Contact
5. Display All Contacts
6. Exit
Invalid choice! Try again.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. Update Contact
4. Delete Contact
5. Display All Contacts
6. Exit
Invalid choice! Try again.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Cont

## Task 15 | Integrated Order Fulfillment System

Concepts: OOP, composition, files, exceptions, algorithms

Problem: Design Product, Cart, Order, and Inventory classes. Read a product catalog from
JSON, build carts, apply percentage discounts, compute totals, and place orders with stock
updates. Persist orders to JSON.

Requirements: Reject invalid quantities and out-of-stock purchases. Validate all items before
changing inventory; roll back if order persistence fails. Generate unique order IDs. Include at least
three unit tests using unittest.

Sample / expected behavior: Catalog: P1 price=100 stock=5; cart P1 x2 with 10% discount ->
total 180, stock becomes 3 after successful order.

In [1]:
# Task 15: Integrated Order Fulfillment System

import json
import os
from uuid import uuid4


class Product:
    def __init__(self, product_id, name, price):
        if price < 0:
            raise ValueError("Price cannot be negative.")
        self.product_id = product_id
        self.name = name
        self.price = price


class Inventory:
    def __init__(self, stock):
        self.stock = stock.copy()

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)
        return cls({item["product_id"]: item["stock"] for item in data})

    def check(self, product_id, quantity):
        if quantity <= 0:
            raise ValueError("Quantity must be greater than zero.")
        if product_id not in self.stock:
            raise ValueError("Product does not exist in inventory.")
        if self.stock[product_id] < quantity:
            raise ValueError("Not enough stock.")

    def reduce(self, product_id, quantity):
        self.stock[product_id] -= quantity


class Cart:
    def __init__(self):
        self.items = {}

    def add(self, product, quantity):
        if quantity <= 0:
            raise ValueError("Quantity must be greater than zero.")
        self.items[product.product_id] = (
            product, self.items.get(product.product_id, (product, 0))[1] + quantity
        )

    def total(self, discount_percent=0):
        if not 0 <= discount_percent <= 100:
            raise ValueError("Discount must be between 0 and 100.")
        subtotal = sum(product.price * quantity
                       for product, quantity in self.items.values())
        return subtotal * (1 - discount_percent / 100)


class Order:
    def __init__(self, cart, discount_percent=0):
        self.order_id = str(uuid4())
        self.items = cart.items.copy()
        self.discount_percent = discount_percent
        self.total = cart.total(discount_percent)

    def to_dict(self):
        return {
            "order_id": self.order_id,
            "items": [
                {
                    "product_id": product.product_id,
                    "name": product.name,
                    "price": product.price,
                    "quantity": quantity
                }
                for product, quantity in self.items.values()
            ],
            "discount_percent": self.discount_percent,
            "total": self.total
        }


def place_order(cart, inventory, orders_file, discount_percent=0):
    if not cart.items:
        raise ValueError("Cart is empty.")

    # Validate every item before changing any stock.
    for product_id, (product, quantity) in cart.items.items():
        inventory.check(product_id, quantity)

    order = Order(cart, discount_percent)

    # Keep old stock so we can roll back if saving fails.
    old_stock = inventory.stock.copy()

    try:
        for product_id, (product, quantity) in cart.items.items():
            inventory.reduce(product_id, quantity)

        try:
            with open(orders_file, "r", encoding="utf-8") as file:
                orders = json.load(file)
        except FileNotFoundError:
            orders = []

        orders.append(order.to_dict())

        # Write to a temporary file first, then replace the original safely.
        temporary_file = orders_file + ".tmp"
        with open(temporary_file, "w", encoding="utf-8") as file:
            json.dump(orders, file, indent=4)
        os.replace(temporary_file, orders_file)

    except Exception:
        inventory.stock = old_stock
        # Remove an unfinished temporary file if one was created.
        temporary_file = orders_file + ".tmp"
        if os.path.exists(temporary_file):
            os.remove(temporary_file)
        raise

    return order


if __name__ == "__main__":
    # Product catalog example.
    with open("catalog.json", "w", encoding="utf-8") as file:
        json.dump([
            {"product_id": "P1", "name": "Notebook", "price": 100, "stock": 5}
        ], file, indent=4)

    inventory = Inventory.from_json("catalog.json")
    product = Product("P1", "Notebook", 100)

    cart = Cart()
    cart.add(product, 2)

    order = place_order(cart, inventory, "orders.json", 10)
    print("Order total:", order.total)       # 180.0
    print("Remaining stock:", inventory.stock["P1"])  # 3


Order total: 180.0
Remaining stock: 3
